# Ноутбук по подготовке данных UNODC, WPP и географических границ

В данном ноутбуке проводится обработка и согласование данных из нескольких источников для последующего анализа показателей умышленных убийств по странам, полу, возрастным группам и годам.

В рамках работы выполняются:
- загрузка исходных таблиц;
- фильтрация и очистка наблюдений;
- унификация названий стран;
- преобразование данных в согласованный формат;
- формирование итоговых таблиц для анализа и визуализации.

## Входные файлы

Для корректного выполнения ноутбука необходимо предварительно загрузить следующие файлы:

- `wpp.xlsx`
- `wpp_male.xlsx`
- `wpp_female.xlsx`
- `unodc.xlsx`
- `geo_raw.csv`

Скачать их можно в репозитории: https://github.com/adkatkaeva/course-project-analysis-data-visualization-based-on-unodc-data
## Выходные файлы

В результате выполнения ноутбука будут созданы следующие файлы:

- `unodc_main.csv`
- `wpp_population.csv`
- `geo_borders.csv`

# Подготовка окружения

В этой части подключаются основные библиотеки для обработки данных:

- `pandas` — для чтения, преобразования и сохранения таблиц;
- `numpy` — для работы с числовыми значениями и пропусками.

Также задаются имена итоговых файлов, которые будут сохранены после обработки данных.

In [47]:
import pandas as pd
import numpy as np

OUT_UNODC = "unodc_main.csv"
OUT_WPP = "wpp_population.csv"
OUT_BORDERS = "geo_borders.csv"

# Сопоставление названий стран

Данные из разных источников используют разные варианты названий стран.  
Чтобы в дальнейшем корректно объединять таблицы, задаются словари соответствий между названиями стран в источниках **UNODC** и **WPP**.

Это нужно для того, чтобы, например, одна и та же страна не воспринималась как две разные записи только из-за различий в написании.

In [48]:
UNODC_TO_WPP = {
    "China, Hong Kong Special Administrative Region": "China, Hong Kong SAR",
    "China, Macao Special Administrative Region": "China, Macao SAR",
    "Iraq (Central Iraq)": "Iraq",
    "Kosovo under UNSCR 12414": "Kosovo (under UNSC res. 1244)",
    "Micronesia (Federated States of)": "Micronesia (Fed. States of)",
    "Netherlands (Kingdom of the)": "Netherlands",
}

WPP_TO_UNODC = {v: k for k, v in UNODC_TO_WPP.items()}

# Функция обработки данных WPP

В этом блоке определяется функция `process_wpp_file`, которая:

1. считывает исходный файл WPP;
2. отбирает только страны и нужные годы;
3. формирует возрастные группы;
4. приводит названия стран к единому стандарту;
5. переводит таблицу в длинный формат.

Под длинным форматом понимается структура, в которой одна строка соответствует одной комбинации:

страна + год + пол + возрастная группа

а значение населения хранится в отдельном столбце.

In [49]:
def process_wpp_file(filepath, sex_label):
    if filepath.endswith('.csv'):
        df_raw = pd.read_csv(filepath, header=1)
        if "Type" not in df_raw.columns:
            df_raw = pd.read_csv(filepath, header=0)
    else:
        df_raw = pd.read_excel(filepath, sheet_name=0, header=1)
        if "Type" not in df_raw.columns:
            df_raw = pd.read_excel(filepath, sheet_name=0, header=0)

    df = df_raw[
        (df_raw["Type"] == "Country/Area") &
        (df_raw["Year"].between(2019, 2023))
    ].copy()

    df.rename(
        columns={
            "Region, subregion, country or area *": "country_wpp",
            "Year": "year"
        },
        inplace=True
    )

    df["pop_Все"] = df["Total"]
    df["pop_0-9"] = df["0-4"] + df["5-14"] / 2
    df["pop_10-14"] = df["5-14"] / 2
    df["pop_15-17"] = df["15-17"]
    df["pop_18-19"] = df["0-19"] - df["0-17"]
    df["pop_20-24"] = df["0-24"] - df["0-19"]
    df["pop_25-29"] = df["25-49"] / 5
    df["pop_30-44"] = df["25-49"] * 3 / 5
    df["pop_45-59"] = df["25-49"] / 5 + (df["50+"] - df["60+"])
    df["pop_60 и старше"] = df["60+"]

    df["country_en"] = df["country_wpp"].map(WPP_TO_UNODC).fillna(df["country_wpp"])

    pop_cols = {
        "Все": "pop_Все",
        "0-9": "pop_0-9",
        "10-14": "pop_10-14",
        "15-17": "pop_15-17",
        "18-19": "pop_18-19",
        "20-24": "pop_20-24",
        "25-29": "pop_25-29",
        "30-44": "pop_30-44",
        "45-59": "pop_45-59",
        "60 и старше": "pop_60 и старше"
    }

    parts = []
    for age_group, col in pop_cols.items():
        if col in df.columns:
            tmp = df[["country_en", "year", col]].copy()
            tmp.columns = ["country_en", "year", "pop_thousands"]
            tmp["age_group"] = age_group
            tmp["sex"] = sex_label
            parts.append(tmp)

    return pd.concat(parts, ignore_index=True).dropna(subset=["pop_thousands"])

# Загрузка данных WPP по полу

Здесь функция применяется к трем отдельным файлам:

- для всего населения;
- для мужчин;
- для женщин.

В результате получаются три таблицы одинаковой структуры, которые затем можно объединить в один набор данных.

In [50]:
wpp_total = process_wpp_file("wpp.xlsx", "Все")
wpp_male = process_wpp_file("wpp_male.xlsx", "Мужчины")
wpp_female = process_wpp_file("wpp_female.xlsx", "Женщины")

# Объединение данных WPP

На этом этапе таблицы для всех, мужчин и женщин объединяются в один датафрейм.

После этого:
- округляются значения численности населения;
- создается словарь с населением United Kingdom по годам, полу и возрастным группам.

Этот словарь далее понадобится для корректного пересчета показателей после объединения частей Великобритании.

In [51]:
df_wpp_long = pd.concat([wpp_total, wpp_male, wpp_female], ignore_index=True)
df_wpp_long["pop_thousands"] = df_wpp_long["pop_thousands"].round(3)

uk_pop_lookup = (
    df_wpp_long[df_wpp_long["country_en"] == "United Kingdom"]
    .set_index(["year", "sex", "age_group"])["pop_thousands"]
    .to_dict()
)

In [52]:
print("Размер WPP (все):", wpp_total.shape)
print("Размер WPP (мужчины):", wpp_male.shape)
print("Размер WPP (женщины):", wpp_female.shape)
print("Размер объединённого WPP:", df_wpp_long.shape)
print("Количество стран в WPP:", df_wpp_long["country_en"].nunique())
print("Годы в WPP:", sorted(df_wpp_long["year"].unique()))
print("Возрастные группы в WPP:", sorted(df_wpp_long["age_group"].unique()))
print("Категории пола в WPP:", sorted(df_wpp_long["sex"].unique()))

Размер WPP (все): (11850, 5)
Размер WPP (мужчины): (11850, 5)
Размер WPP (женщины): (11850, 5)
Размер объединённого WPP: (35550, 5)
Количество стран в WPP: 237
Годы в WPP: [np.float64(2019.0), np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0)]
Возрастные группы в WPP: ['0-9', '10-14', '15-17', '18-19', '20-24', '25-29', '30-44', '45-59', '60 и старше', 'Все']
Категории пола в WPP: ['Все', 'Женщины', 'Мужчины']


# Загрузка данных UNODC

В этом блоке считывается исходный Excel-файл с данными по умышленным убийствам.

Далее из него будут выбраны только нужные столбцы и записи, относящиеся к исследуемому периоду и нужному показателю.

In [53]:
df_raw = pd.read_excel("unodc.xlsx")

# Переименование столбцов

Поскольку исходная таблица может содержать русские и английские названия столбцов, здесь выполняется их автоматическое переименование в единый рабочий формат.

Это делает код более понятным и удобным для последующей обработки.

In [54]:
col_rename = {}
for col in df_raw.columns:
    if "Код ISO-3" in str(col):
        col_rename[col] = "iso3"
    elif "Страна" in str(col) or "Название страны" in str(col):
        col_rename[col] = "country_ru"
    elif "Country" in str(col):
        col_rename[col] = "Country"
    elif "Регион" in str(col):
        col_rename[col] = "region_ru"
    elif "Region" in str(col):
        col_rename[col] = "Region"
    elif "Субрегион" in str(col):
        col_rename[col] = "subregion_ru"
    elif "Subregion" in str(col):
        col_rename[col] = "Subregion"
    elif "Год" in str(col):
        col_rename[col] = "year"
    elif "Пол" in str(col):
        col_rename[col] = "sex"
    elif "Возраст" in str(col):
        col_rename[col] = "age_raw"
    elif "Показатель" in str(col):
        col_rename[col] = "indicator"
    elif "Характер" in str(col):
        col_rename[col] = "nature"
    elif "Единица измерения" in str(col):
        col_rename[col] = "unit"
    elif "Категория" in str(col):
        col_rename[col] = "category"
    elif "Значение" in str(col):
        col_rename[col] = "value"

df_raw.rename(columns=col_rename, inplace=True)

# Фильтрация наблюдений

На этом этапе из исходной таблицы оставляются только записи, удовлетворяющие условиям:

- показатель: жертвы умышленного убийства;
- характер: все;
- категория: все;
- годы: с 2019 по 2023.

Дополнительно очищаются названия возрастных групп и исключаются записи с неизвестным возрастом.

In [55]:
df = df_raw[
    (df_raw["indicator"] == "Жертвы умышленного убийства") &
    (df_raw["nature"] == "Все") &
    (df_raw["category"] == "Все") &
    (df_raw["year"].between(2019, 2023))
].copy()

df["age_group"] = (
    df["age_raw"]
    .astype(str)
    .str.strip()
    .str.replace("10 -14", "10-14")
    .str.replace("15 -17", "15-17")
)

df = df[df["age_group"] != "Неизвестно"].copy()

In [56]:
print("Размер отфильтрованного UNODC:", df.shape)
print("Годы в UNODC:", sorted(df["year"].unique()))
print("Возрастные группы в UNODC:", sorted(df["age_group"].unique()))
print("Категории пола в UNODC:", sorted(df["sex"].dropna().unique()))
print("Количество стран в UNODC:", df["Country"].nunique())

Размер отфильтрованного UNODC: (13232, 17)
Годы в UNODC: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Возрастные группы в UNODC: ['0-9', '10-14', '15-17', '18-19', '20-24', '25-29', '30-44', '45-59', '60 и старше', 'Все']
Категории пола в UNODC: ['Все', 'Женщина', 'Мужчина']
Количество стран в UNODC: 152


# Нормализация регионов

Здесь русские названия регионов приводятся к английскому виду.

Это нужно для единообразия итоговой таблицы и для упрощения последующего анализа и визуализации данных.

In [57]:
region_mapping = {
    "Азия": "Asia",
    "Америка": "Americas",
    "Африка": "Africa",
    "Европа": "Europe",
    "Океания": "Oceania"
}

df["Region"] = df["region_ru"].map(region_mapping).fillna(df["region_ru"])
df["Subregion"] = df["subregion_ru"]

# Разделение показателей на количество и уровень

В исходных данных присутствуют два типа значений:

- абсолютное количество жертв;
- показатель на 100 000 населения.

В этом блоке они разделяются на две отдельные таблицы, чтобы затем их можно было аккуратно объединить по ключевым признакам.

In [58]:
df_count = df[df["unit"] == "Количество"][
    [
        "iso3", "country_ru", "Country", "region_ru", "Region",
        "subregion_ru", "Subregion", "year", "sex", "age_group", "value"
    ]
].rename(columns={"value": "count"})

df_rate = df[df["unit"] == "На 100 000 населения"][
    ["year", "sex", "age_group", "Country", "value"]
].rename(columns={"value": "rate_per_100k"})

# Объединение count и rate

После разделения данных создается единая таблица, где для каждой комбинации признаков доступны:

- абсолютное число жертв;
- уровень на 100 000 населения.

Также числовые столбцы приводятся к числовому типу, а обозначения пола нормализуются.

In [59]:
df_wide = df_count.merge(
    df_rate,
    on=["year", "sex", "age_group", "Country"],
    how="outer"
)

df_wide["count"] = pd.to_numeric(df_wide["count"], errors="coerce")
df_wide["rate_per_100k"] = pd.to_numeric(df_wide["rate_per_100k"], errors="coerce")

df_wide["sex"] = df_wide["sex"].replace({
    "Мужчина": "Мужчины",
    "Женщина": "Женщины"
})

# Выделение частей Великобритании

В данных UNODC части Великобритании могут идти отдельно, например:

- England and Wales;
- Northern Ireland;
- Scotland.

Поэтому на данном шаге они отделяются от остальных стран, чтобы затем объединить их в одну запись `United Kingdom`.

In [60]:
UK_PARTS = [
    "United Kingdom (England and Wales)",
    "United Kingdom (Northern Ireland)",
    "United Kingdom (Scotland)"
]

df_uk = df_wide[df_wide["Country"].isin(UK_PARTS)].copy()
df_non_uk = df_wide[~df_wide["Country"].isin(UK_PARTS)].copy()

# Функция объединения United Kingdom

В этом блоке определяется функция `merge_uk`, которая агрегирует части Великобритании в одну страну.

Логика следующая:

- абсолютные значения суммируются;
- показатель на 100 000 населения пересчитывается через численность населения из WPP.

Формула пересчета имеет вид:
$$
\text{R} = \frac{\text{count}}{\text{population}} \cdot 100000
$$

где население предварительно переводится из тысяч человек в человек.

In [61]:
def merge_uk(df_uk, uk_pop_lookup):
    groups = df_uk.groupby(["year", "sex", "age_group"])
    rows = []

    for (year, sex, age), grp in groups:
        total_count = grp["count"].sum(skipna=True)
        pop_thousands = uk_pop_lookup.get((year, sex, age), np.nan)

        if pd.notna(pop_thousands) and pop_thousands > 0:
            rate_uk = (total_count / (pop_thousands * 1000)) * 100_000
        else:
            rate_uk = grp["rate_per_100k"].mean(skipna=True)

        rows.append({
            "iso3": "GBR",
            "country_ru": "Великобритания",
            "Country": "United Kingdom",
            "region_ru": "Европа",
            "Region": "Europe",
            "subregion_ru": "Северная Европа",
            "Subregion": "Northern Europe",
            "year": year,
            "sex": sex,
            "age_group": age,
            "count": total_count,
            "rate_per_100k": round(rate_uk, 6) if pd.notna(rate_uk) else np.nan,
            "uk_partial": (age != "Все")
        })

    return pd.DataFrame(rows)

# Формирование итоговой таблицы UNODC

Здесь объединяются:

- все страны, которые не требуют специальной обработки;
- агрегированная запись для `United Kingdom`.

После этого получается итоговый набор данных по жертвам умышленных убийств в согласованном формате.

In [62]:
df_uk_merged = merge_uk(df_uk, uk_pop_lookup)
df_result = pd.concat([df_non_uk, df_uk_merged], ignore_index=True)

if "uk_partial" in df_result.columns:
    df_result.drop(columns=["uk_partial"], inplace=True)

In [63]:
print("Размер итоговой таблицы UNODC:", df_result.shape)
print("Количество стран в итоговой таблице:", df_result["Country"].nunique())
print("Годы в итоговой таблице:", sorted(df_result["year"].unique()))
print("Возрастные группы в итоговой таблице:", sorted(df_result["age_group"].unique()))
print("Количество строк для United Kingdom:", (df_result["Country"] == "United Kingdom").sum())

Размер итоговой таблицы UNODC: (6540, 12)
Количество стран в итоговой таблице: 150
Годы в итоговой таблице: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Возрастные группы в итоговой таблице: ['0-9', '10-14', '15-17', '18-19', '20-24', '25-29', '30-44', '45-59', '60 и старше', 'Все']
Количество строк для United Kingdom: 87


# Приведение названий столбцов WPP

В этом блоке столбец с названием страны в данных WPP переименовывается так, чтобы его имя совпадало с названием столбца в данных UNODC.

Это упрощает дальнейшие сравнения и проверки между наборами данных.

In [64]:
df_wpp_long.rename(columns={"country_en": "Country"}, inplace=True)

# Загрузка географических данных

На этом шаге считывается файл с границами стран.

Из него оставляются только записи, содержащие полигоны границ государств, которые будут использоваться в итоговом географическом наборе данных.

In [65]:
df_borders = pd.read_csv("geo_raw.csv", sep=";")
df_borders = df_borders[
    df_borders["coords_type"] == "all_country_borders_poly"
].copy()

# Приведение названий стран в геоданных

Как и в предыдущих источниках, в географическом файле названия стран могут отличаться.

Поэтому выполняется замена названий на единый формат, совместимый с данными UNODC.  
После этого выбираются только нужные столбцы: страна, тип координат и сами координаты.

In [66]:
BORDERS_TO_UNODC = {
    "The Bahamas": "Bahamas",
    "Bolivia": "Bolivia (Plurinational State of)",
    "Cape Verde": "Cabo Verde",
    "Iraq": "Iraq (Central Iraq)",
    "Kosovo": "Kosovo under UNSCR 1244",
    "Federated States of Micronesia": "Micronesia (Federated States of)",
    "Netherlands": "Netherlands (Kingdom of the)",
    "South Korea": "Republic of Korea",
    "Moldova": "Republic of Moldova",
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Tanzania": "United Republic of Tanzania",
    "United States": "United States of America",
    "Venezuela": "Venezuela (Bolivarian Republic of)",
    "Brunei": "Brunei Darussalam",
    "Congo-Brazzaville": "Congo",
    "East Timor": "Timor-Leste",
    "Iran": "Iran (Islamic Republic of)",
    "Laos": "Lao People's Democratic Republic",
    "North Korea": "Dem. People's Republic of Korea",
    "Syria": "Syrian Arab Republic",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "The Gambia": "Gambia",
    "Vietnam": "Viet Nam",
    "Vatican City": "Holy See",
}

df_borders["Country"] = df_borders["name_en"].replace(BORDERS_TO_UNODC)

df_borders_out = df_borders[["Country", "coords_type", "coords"]].copy()

# Сохранение итоговых файлов

В этой части формируются и сохраняются три итоговых файла:

- основной набор данных UNODC;
- таблица населения WPP;
- таблица с географическими границами.

Все файлы сохраняются в формате CSV с кодировкой `utf-8-sig`, что удобно для последующего открытия в разных программах.

In [67]:
df_result.to_csv(OUT_UNODC, index=False, encoding="utf-8-sig")
df_wpp_long.to_csv(OUT_WPP, index=False, encoding="utf-8-sig")
df_borders_out.to_csv(OUT_BORDERS, index=False, encoding="utf-8-sig")

# Проверка согласованности данных

Последний блок можно использовать для диагностики качества подготовки данных.

Здесь можно проверить:
- какие страны присутствуют в одном источнике, но отсутствуют в другом;
- совпадают ли возрастные группы между таблицами;
- нет ли проблем с унификацией названий.

Этот этап полезен для контроля корректности объединения данных из нескольких источников.

In [68]:
unodc_countries = set(df_result["Country"].dropna().unique())
wpp_countries = set(df_wpp_long["Country"].dropna().unique())
geo_countries = set(df_borders_out["Country"].dropna().unique())

unodc_not_in_wpp = sorted(unodc_countries - wpp_countries)
wpp_not_in_unodc = sorted(wpp_countries - unodc_countries)
unodc_not_in_geo = sorted(unodc_countries - geo_countries)
geo_not_in_unodc = sorted(geo_countries - unodc_countries)

unodc_age_groups = sorted(df["age_group"].unique())
wpp_age_groups = sorted(df_wpp_long["age_group"].unique())

In [69]:
print("Есть в UNODC, но нет в WPP:", len(unodc_not_in_wpp), unodc_not_in_wpp[:10])
print("Есть в WPP, но нет в UNODC:", len(wpp_not_in_unodc), wpp_not_in_unodc[:10])
print("Есть в UNODC, но нет в GEO:", len(unodc_not_in_geo), unodc_not_in_geo[:10])
print("Есть в GEO, но нет в UNODC:", len(geo_not_in_unodc), geo_not_in_unodc[:10])

print("Возрастные группы в UNODC:", unodc_age_groups)
print("Возрастные группы в WPP:", wpp_age_groups)

Есть в UNODC, но нет в WPP: 1 ['Kosovo under UNSCR 1244']
Есть в WPP, но нет в UNODC: 88 ['Angola', 'Anguilla', 'Aruba', 'Bangladesh', 'Benin', 'Bonaire, Sint Eustatius and Saba', 'British Virgin Islands', 'Brunei Darussalam', 'Burkina Faso', 'Burundi']
Есть в UNODC, но нет в GEO: 8 ['American Samoa', 'China, Hong Kong Special Administrative Region', 'China, Macao Special Administrative Region', 'French Guiana', 'Guam', 'Puerto Rico', 'State of Palestine', 'Tonga']
Есть в GEO, но нет в UNODC: 75 ['Angola', 'Anguilla', 'Bangladesh', 'Benin', 'British Indian Ocean Territory', 'British Sovereign Base Areas', 'British Virgin Islands', 'Brunei Darussalam', 'Burkina Faso', 'Burundi']
Возрастные группы в UNODC: ['0-9', '10-14', '15-17', '18-19', '20-24', '25-29', '30-44', '45-59', '60 и старше', 'Все']
Возрастные группы в WPP: ['0-9', '10-14', '15-17', '18-19', '20-24', '25-29', '30-44', '45-59', '60 и старше', 'Все']
